![image.png](https://i.imgur.com/4fN73lZ.png)

This notebook has been inspired from [PPO](https://github.com/nikhilbarhate99/PPO-PyTorch) by Nikhil Barhate, [pytorch_ppo_lunarlander_v2](https://github.com/lucaslingle/pytorch_ppo_lunarlander_v2/tree/main) by Lucas Lingle and [PPO OpenAI SpinningUp](https://spinningup.openai.com/en/latest/algorithms/ppo.html).

### Setup

We standardize on **Gymnasium** (with Box2D for LunarLander) plus `imageio` for GIF rendering, installed with **uv**. `torch` is the deep-learning backend.

In [ ]:
# Fast dependency install with uv (https://docs.astral.sh/uv).
# Bootstraps uv via pip, then installs into the current kernel/venv. If you already
# run from the project's .venv (created with `uv sync`), this is essentially a no-op.
# (torch ships with Colab; locally `uv sync` installs it.)
import sys, os
%pip install -q uv
_target = "" if (sys.prefix != sys.base_prefix or os.environ.get("VIRTUAL_ENV")) else "--system"
!uv pip install -q {_target} --python "{sys.executable}" swig
!uv pip install -q {_target} --python "{sys.executable}" "gymnasium[box2d]" imageio matplotlib torch

# PPO

In this notebook, we will implementa basic PPO Reinforcement learning algorithm for Lunar Lander Environment.

## Lunar Lander

This environment is a classic rocket trajectory optimization problem. The landing pad is always at coordinates (0,0). The state is an 8-dimensional vector: the coordinates of the lander in x & y, its linear velocities in x & y, its angle, its angular velocity, and two booleans that represent whether each leg is in contact with the ground or not.

There are four discrete actions available:<br>
- 0: do nothing<br>
- 1: fire left orientation engine<br>
- 2: fire main engine<br>
- 3: fire right orientation engine<br>

After every step a reward is granted. The total reward of an episode is the sum of the rewards for all the steps within that episode.

For each step, the reward:

- is increased/decreased the closer/further the lander is to the landing pad.

- is increased/decreased the slower/faster the lander is moving.

- is decreased the more the lander is tilted (angle not horizontal).

- is increased by 10 points for each leg that is in contact with the ground.

- is decreased by 0.03 points each frame a side engine is firing.

- is decreased by 0.3 points each frame the main engine is firing.

The episode receive an additional reward of -100 or +100 points for crashing or landing safely respectively.

An episode is considered a solution if it scores at least 200 points.


You can read more the cartpole environment [here](https://gymnasium.farama.org/environments/box2d/lunar_lander/)

![Cartpole](https://gymnasium.farama.org/_images/lunar_lander.gif)

## PPO

PPO is motivated by the same question as TRPO: how can we take the biggest possible improvement step on a policy using the data we currently have, without stepping so far that we accidentally cause performance collapse? Where TRPO tries to solve this problem with a complex second-order method, PPO is a family of first-order methods that use a few other tricks to keep new policies close to old. PPO methods are significantly simpler to implement, and empirically seem to perform at least as well as TRPO.

There are two primary variants of PPO: PPO-Penalty and PPO-Clip.

**PPO-Penalty** approximately solves a KL-constrained update like TRPO, but penalizes the KL-divergence in the objective function instead of making it a hard constraint, and automatically adjusts the penalty coefficient over the course of training so that it's scaled appropriately.

**PPO-Clip** doesn't have a KL-divergence term in the objective and doesn’t have a constraint at all. Instead relies on specialized clipping in the objective function to remove incentives for the new policy to get far from the old policy.

Read more [here](https://spinningup.openai.com/en/latest/algorithms/ppo.html).

Here, we'll focus only on PPO-Clip.

![ppo_2.png](https://i.imgur.com/VCKH7yN.png)

[Image Source](http://rail.eecs.berkeley.edu/deeprlcourse-fa17/)

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# Create the environment
env = gym.make("LunarLander-v3", render_mode="rgb_array")

### The PPO agent

The agent bundles the actor-critic network with the PPO logic. Its methods, in the order the
training loop uses them:

- `select_action(state)` — sample an action and remember its log-prob (the "old" policy, for the ratio)
- `compute_returns(rewards, dones)` — discounted returns over the rollout
- `compute_advantages(states, returns)` — advantage = return − V(s), normalised
- `ppo_loss(...)` — the **clipped surrogate** objective (the heart of PPO)
- `update(...)` — reuse the rollout for `k_epochs` gradient steps on `ppo_loss`

A `Rollout` just stores the transitions collected under the current policy for one update.

In [ ]:
# Actor-critic network: a shared body with a policy head (actor) and a value head (critic).
class ActorCritic(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 32)
        self.actor = nn.Linear(32, output_dim)   # action logits
        self.critic = nn.Linear(32, 1)           # state value V(s)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.actor(x), self.critic(x)


# A rollout simply stores the transitions collected under the current policy for one update.
class Rollout:
    def __init__(self):
        self.clear()

    def clear(self):
        self.states, self.actions, self.rewards, self.dones, self.log_probs = [], [], [], [], []

    def add(self, state, action, reward, done, log_prob):
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)
        self.dones.append(done)
        self.log_probs.append(log_prob)

In [ ]:
class PPOAgent:
    def __init__(self, input_dim, output_dim, lr=1e-3, gamma=0.99, epsilon=0.2,
                 k_epochs=4, device="cpu"):
        self.actor_critic = ActorCritic(input_dim, output_dim).to(device)
        self.optimizer = optim.Adam(self.actor_critic.parameters(), lr=lr)
        self.gamma = gamma
        self.epsilon = epsilon
        self.k_epochs = k_epochs
        self.device = device

    def select_action(self, state):
        """Sample an action under the current policy; return (action, log_prob).
        The log_prob is kept as the OLD log-prob for the PPO ratio."""
        state = torch.tensor(np.asarray(state), dtype=torch.float32, device=self.device)
        with torch.no_grad():
            logits, _ = self.actor_critic(state)
            dist = Categorical(F.softmax(logits, dim=-1))
            action = dist.sample()
        return action.item(), dist.log_prob(action)

    def compute_returns(self, rewards, dones):
        """Discounted returns over the rollout, reset at episode boundaries (dones)."""
        returns, R = [], 0.0
        for r, done in zip(reversed(rewards), reversed(dones)):
            if done:
                R = 0.0
            R = r + self.gamma * R
            returns.insert(0, R)
        return torch.tensor(returns, dtype=torch.float32, device=self.device)

    def compute_advantages(self, states, returns):
        """Advantage = return - V(s) (the critic as a baseline), normalised for stable gradients."""
        with torch.no_grad():
            states_t = torch.tensor(np.array(states), dtype=torch.float32, device=self.device)
            values = self.actor_critic(states_t)[1].squeeze(-1)
        advantages = returns - values
        return (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    def ppo_loss(self, states, actions, returns, advantages, old_log_probs):
        """The PPO-Clip loss for one pass over the rollout:
        clipped policy loss + value loss - entropy bonus."""
        logits, values = self.actor_critic(states)
        dist = Categorical(F.softmax(logits, dim=-1))
        log_probs = dist.log_prob(actions)

        ratio = torch.exp(log_probs - old_log_probs)             # pi_new / pi_old
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1 - self.epsilon, 1 + self.epsilon) * advantages
        policy_loss = -torch.min(surr1, surr2).mean()

        value_loss = F.smooth_l1_loss(values.squeeze(-1), returns)
        entropy = dist.entropy().mean()
        return policy_loss + 0.5 * value_loss - 0.01 * entropy

    def update(self, states, actions, returns, advantages, old_log_probs):
        """Reuse the rollout for k_epochs of gradient steps on the PPO loss."""
        states = torch.tensor(np.array(states), dtype=torch.float32, device=self.device)
        actions = torch.tensor(actions, dtype=torch.int64, device=self.device)
        old_log_probs = torch.stack(old_log_probs).detach().to(self.device)
        for _ in range(self.k_epochs):
            loss = self.ppo_loss(states, actions, returns, advantages, old_log_probs)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

### Hyperparameters

In [ ]:
# Hyperparameters
total_episodes = 1000        # Total training episodes
max_steps = 500              # Max steps per episode
learning_rate = 1e-3         # Optimizer learning rate
gamma = 0.99                 # Discount factor
epsilon = 0.2                # PPO clip range
k_epochs = 10                # SGD epochs per update (reuse the same rollout)
train_freq = 5               # Episodes collected into one rollout before each update

### Training: collecting a rollout

PPO is **on-policy**: it learns from data produced by the *current* policy. Each update uses a
**rollout** — the batch of transitions collected by running the current policy. Here we gather a
rollout spanning `train_freq` episodes, reuse it for `k_epochs` gradient steps, then **discard
it**. (Unlike the Day 2 *replay buffer*, on-policy data cannot be reused once the policy has
changed.) The Flappy Bird lab is the same, with a one-episode rollout.

**Why normalise the returns here?** LunarLander's rewards are large in magnitude (roughly ±100 for crashing / landing, plus shaping terms), so the discounted returns span a wide range. Before computing the loss we normalise them to zero mean / unit variance: this keeps the critic's regression targets and the advantages well-scaled, which stabilises training. Small-reward tasks like the Flappy Bird lab don't need this step.

In [ ]:
print('observation space:', env.observation_space)
print('action space:', env.action_space)

state_size = env.observation_space.shape[0]
action_size = env.action_space.n

agent = PPOAgent(state_size, action_size, lr=learning_rate, gamma=gamma,
                 epsilon=epsilon, k_epochs=k_epochs, device=device)

In [ ]:
scores = []
rollout = Rollout()   # accumulates across episodes (this is the only difference from the Flappy lab)

for episode in range(1, total_episodes + 1):
    state, _ = env.reset()
    ep_reward = 0
    for t in range(max_steps):
        action, log_prob = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        rollout.add(state, action, reward, terminated or truncated, log_prob)
        state = next_state
        ep_reward += reward
        if terminated or truncated:
            break
    scores.append(ep_reward)

    # once the rollout spans train_freq episodes, run the PPO update and start a fresh rollout
    if episode % train_freq == 0:
        returns = agent.compute_returns(rollout.rewards, rollout.dones)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)  # normalise: LunarLander returns are large-scale (see the note above)
        advantages = agent.compute_advantages(rollout.states, returns)
        agent.update(rollout.states, rollout.actions, returns, advantages, rollout.log_probs)
        rollout.clear()

    if episode % 20 == 0:
        print(f"Episode {episode:4d} | avg reward (last 20): {np.mean(scores[-20:]):7.1f}")

In [ ]:
def moving_average(x, window):
    x = np.asarray(x, dtype=float)
    return x if len(x) < window else np.convolve(x, np.ones(window) / window, mode="valid")

plt.figure(figsize=(12, 4))
plt.plot(scores, alpha=0.3, label="per-episode reward")
plt.plot(np.arange(len(scores) - len(moving_average(scores, 20)), len(scores)),
         moving_average(scores, 20), label="moving average (20)")
plt.axhline(200, color="green", ls="--", alpha=0.5, label="solved (200)")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("PPO on LunarLander-v3")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Visualization

In [ ]:
# Visualization helpers (Gymnasium-native, no dependency on the old `gym` package)
import os
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")  # headless rendering (e.g. Colab)
import imageio.v2 as imageio
from IPython.display import Image, display

os.makedirs("video", exist_ok=True)

def record_gif(env, agent, name, max_steps=500, fps=30):
    """Roll out the policy and save the episode as a GIF."""
    frames = []
    state, _ = env.reset()
    for _ in range(max_steps):
        frames.append(env.render())
        action, _, _ = agent.act(state)
        state, reward, terminated, truncated, _ = env.step(action.item())
        if terminated or truncated:
            break
    env.close()
    path = f"video/{name}.gif"
    imageio.mimsave(path, frames, fps=fps, loop=0)
    return path

def show_gif(name):
    display(Image(filename=f"video/{name}.gif"))

In [ ]:
eval_env = gym.make("LunarLander-v3", render_mode="rgb_array")
record_gif(eval_env, agent, "LunarLander")
show_gif("LunarLander")